# Ingestion Pipeline & Incremental Updates

`VectorStoreIndex.from_documents()` re-chunks and re-embeds every document every time it runs — fine for a 5-document demo, expensive and slow for a real corpus that grows over time. `IngestionPipeline` fixes this: it caches chunking/embedding work per-document and, with a docstore attached, skips documents it has already processed, embedding only what's actually new or changed.


**Step 1 — Setup.** Configure logging, load API keys, set the global LLM/embedding models, and clear out any leftover Chroma data from a previous run so this notebook behaves the same way every time it's executed.


In [1]:
import logging
import shutil

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down the noisy INFO-level logs from httpx and llama_index.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Load API keys from .env into the environment.
load_dotenv()

# Global defaults used by every index/pipeline built in this notebook.
Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Clean slate each run so this notebook stays independently runnable
shutil.rmtree("chroma_ingestion_demo", ignore_errors=True)

**Step 2 — Build the pipeline.** Wire up a Chroma vector store (to hold the embedded chunks) and a `SimpleDocumentStore` (to remember which documents have already been processed), then create an `IngestionPipeline` that chunks and embeds documents through those two stores.


In [2]:
import chromadb
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

# A persistent Chroma client + collection is where the pipeline's embedded
# chunks will actually be stored on disk.
chroma_client = chromadb.PersistentClient(path="chroma_ingestion_demo")
collection = chroma_client.get_or_create_collection("ingestion_demo")
vector_store = ChromaVectorStore(chroma_collection=collection)
docstore = SimpleDocumentStore()  # tracks which documents have already been processed

# transformations run in order on every document: split into 512-token chunks,
# then embed each chunk. The docstore + vector_store let the pipeline detect
# and skip documents it has already seen on future runs.
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=512),
        OpenAIEmbedding(model="text-embedding-3-small"),
    ],
    docstore=docstore,
    vector_store=vector_store,
)

**Step 3 — Run it twice on the same documents.** The first run has to chunk and embed everything from scratch. The second run passes in the exact same documents — watch the timing and node count to see the docstore skip work it's already done.


In [3]:
import time

from llama_index.core import SimpleDirectoryReader

# Load the same five anime documents used throughout this series.
documents = SimpleDirectoryReader("data/sample_docs").load_data()

# First run: nothing is cached yet, so every document gets chunked and embedded.
start = time.perf_counter()
nodes = pipeline.run(documents=documents)
elapsed = time.perf_counter() - start
print(f"First run: processed {len(nodes)} new nodes in {elapsed:.2f}s")

# Second run with the identical documents: the docstore recognizes each one by
# its content hash and skips re-chunking/re-embedding entirely.
start = time.perf_counter()
nodes_again = pipeline.run(documents=documents)
elapsed = time.perf_counter() - start
print(f"Second run, same documents: processed {len(nodes_again)} new nodes in {elapsed:.2f}s")

First run: processed 17 new nodes in 9.79s
Second run, same documents: processed 0 new nodes in 0.00s


**Step 4 — Add one new document.** Now grow the corpus by one document (a One Piece overview) and run the pipeline again on the full set — the docstore should only embed the single new document, not reprocess the five it already knows about.


In [4]:
import time
from llama_index.core import Document

# A brand-new Document, built directly instead of read from a file, with an
# explicit doc_id so the docstore can track it individually.
new_doc = Document(
    text=(
        "One Piece — Series Overview\n\n"
        "One Piece is a long-running pirate adventure manga created by Eiichiro Oda, "
        "following Monkey D. Luffy's crew as they search for the legendary treasure "
        "One Piece. It is serialized in Weekly Shonen Jump and animated by Toei Animation."
    ),
    doc_id="one_piece_note",
)

# Pass in the original five documents plus the new one — the pipeline diffs
# against the docstore and only processes what it hasn't seen before.
start = time.perf_counter()
nodes_updated = pipeline.run(documents=documents + [new_doc])
elapsed = time.perf_counter() - start
print(f"After adding 1 new document: processed {len(nodes_updated)} new node(s) in {elapsed:.2f}s")
print(f"Docstore now tracks {len(docstore.docs)} documents total")

After adding 1 new document: processed 1 new node(s) in 0.80s
Docstore now tracks 6 documents total


**Step 5 — Query without re-indexing.** The pipeline's `vector_store` already holds every embedded chunk, old and new — build a `VectorStoreIndex` straight from it and confirm the new document is actually queryable.


In [7]:
from llama_index.core import VectorStoreIndex

# The vector_store attached to the pipeline already has everything embedded —
# building an index from it needs no re-embedding at all.
index = VectorStoreIndex.from_vector_store(vector_store)
response = index.as_query_engine().query("Who created One Piece, and who animates it?")
print(response)

One Piece was created by Eiichiro Oda, and it is animated by Toei Animation.


### Summary

- `IngestionPipeline` with a `docstore` attached skips documents it has already processed — the second run above did zero embedding work for unchanged documents.
- Adding one new document to a five-document corpus embeds exactly one document's worth of nodes, not the whole corpus again — this is what makes incremental updates to a growing knowledge base practical.
- The pipeline's `vector_store` is a normal LlamaIndex vector store underneath — `VectorStoreIndex.from_vector_store()` queries it directly with no separate re-indexing step.
